In [ ]:
# Importar bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Carregar métricas
arima = pd.read_csv('https://raw.githubusercontent.com/cleodecker/TCC/refs/heads/main/M%C3%A9tricas/ARIMA_modelos_metricas_arima.csv')
ets = pd.read_csv("https://raw.githubusercontent.com/cleodecker/TCC/refs/heads/main/M%C3%A9tricas/ETS_metricas.csv")
lc = pd.read_csv("https://raw.githubusercontent.com/cleodecker/TCC/refs/heads/main/M%C3%A9tricas/LC_metricas_por_sexo.csv")
fdm = pd.read_csv("https://raw.githubusercontent.com/cleodecker/TCC/refs/heads/main/M%C3%A9tricas/FDM_metricas_por_sexo.csv")
tl_cnn_gru = pd.read_csv("https://raw.githubusercontent.com/cleodecker/TCC/refs/heads/main/M%C3%A9tricas/transfer_cnn_gru_metricas.csv")
tl_cnn = pd.read_csv("https://raw.githubusercontent.com/cleodecker/TCC/refs/heads/main/M%C3%A9tricas/transfer_cnn_metricas.csv")
tl_gru = pd.read_csv("https://raw.githubusercontent.com/cleodecker/TCC/refs/heads/main/M%C3%A9tricas/transfer_gru_metricas.csv")
media_simples = pd.read_csv("https://raw.githubusercontent.com/cleodecker/TCC/refs/heads/main/M%C3%A9tricas/metrica_media_simples.csv")
ponderada = pd.read_csv("https://raw.githubusercontent.com/cleodecker/TCC/refs/heads/main/M%C3%A9tricas/metricas_previsoes_combinadas_ponder.csv")
ponderada

In [ ]:
# arima, ets, lc, fdm, tl_cnn, tl_gru, tl_cnn_gru, media_simples
# colocar todos em um dicionário para facilitar
dfs = {
    "ARIMA": arima,
    "ETS": ets,
    "LC": lc,
    "FDM": fdm,
    "TL_CNN": tl_cnn,
    "TL_GRU": tl_gru,
    "TL_CNN_GRU": tl_cnn_gru,
    "COMB_MEDIA": media_simples,
    "COMB_PONDERADA": ponderada
}

sexos = ["Feminino", "Masculino", "Ambos"]
metricas = ["RMSE", "MAE", "sMAPE"]

# dicionário para armazenar os 9 dataframes finais
resultados = {}

for sexo in sexos:
    for metrica in metricas:
        merged = None
        for nome, df in dfs.items():
            df_sel = df[df["sexo"] == sexo][["idade", metrica]].rename(columns={metrica: nome})
            if merged is None:
                merged = df_sel
            else:
                merged = pd.merge(merged, df_sel, on="idade", how="outer")
        resultados[(sexo, metrica)] = merged.sort_values("idade").reset_index(drop=True)

# Exemplo de acesso:
df_fem_rmse = resultados[("Feminino", "RMSE")]
print(df_fem_rmse)


In [ ]:
df_fem_mae = resultados[("Feminino", "MAE")]
print(df_fem_mae)

In [ ]:
df_fem_smap = resultados[("Feminino", "sMAPE")]
print(df_fem_smap)

In [ ]:
#Gráficos Boxplot
def plot_boxplot(df, metrica, titulo, log=False):
    # Transformar em formato longo
    df_long = df.melt(id_vars="idade", var_name="Modelo", value_name=metrica)

    # Ordem dos modelos pela média
    ordem = df_long.groupby("Modelo")[metrica].mean().sort_values().index

    # Criar figura
    plt.figure(figsize=(10, 6))
    sns.boxplot(
        data=df_long,
        x="Modelo",
        y=metrica,
        order=ordem,
        palette="viridis"
    )

    if log:
        plt.yscale("log")
        y_label = f"Log({metrica})"
    else:
        y_label = f"{metrica}"

    plt.xlabel("Modelo")
    plt.ylabel(y_label)

    plt.title(titulo, fontsize=14)
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

def gerar_conjunto_graficos(resultados, sexo):
    metricas = ["RMSE", "MAE", "sMAPE"]
    for metrica in metricas:
        log = True if metrica in ["RMSE", "MAE"] else False
        titulo = f"{metrica} - {sexo}"
        df = resultados[(sexo, metrica)]
        plot_boxplot(df, metrica, titulo, log=log)

# ============================
# Exemplo de uso
# ============================

# Gráficos para Feminino
gerar_conjunto_graficos(resultados, "Feminino")

# Gráficos para Masculino
gerar_conjunto_graficos(resultados, "Masculino")

# Gráficos para Ambos
gerar_conjunto_graficos(resultados, "Ambos")

In [ ]:
# Ordem fixa dos modelos
ordem_modelos = ["ARIMA", "ETS", "LC", "FDM", "TL_CNN", "TL_GRU", "TL_CNN_GRU", "COMB_MEDIA", "COMB_PONDERADA"]

# Paleta viridis fixa
palette = sns.color_palette("viridis", len(ordem_modelos))

def plot_barras_por_idade(df, metrica, sexo):
    # Coloca em formato longo
    df_long = df.melt(id_vars="idade", var_name="Modelo", value_name=metrica)

    plt.figure(figsize=(12, 6))
    sns.barplot(
        data=df_long,
        x="idade",
        y=metrica,
        hue="Modelo",
        hue_order=ordem_modelos,
        palette=palette
    )

    # Escala log para RMSE e MAE, linear para sMAPE
    if metrica in ["RMSE", "MAE"]:
        plt.yscale("log")
        y_label = f"Log({metrica})"
    else:
        y_label = f"{metrica}"

    plt.xlabel("Idade")
    plt.ylabel(y_label)

    plt.title(f"{metrica} por idade - {sexo}", fontsize=14)
    plt.legend(title="Modelo", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

def gerar_barras(resultados, sexo):
    for metrica in ["RMSE", "MAE", "sMAPE"]:
        df = resultados[(sexo, metrica)]
        plot_barras_por_idade(df, metrica, sexo)

# ============================
# Exemplo de uso
# ============================

# Feminino
gerar_barras(resultados, "Feminino")

# Masculino
gerar_barras(resultados, "Masculino")

# Ambos
gerar_barras(resultados, "Ambos")



In [ ]:
def medias_por_modelo(df, metrica):
    """
    Calcula a média da métrica por modelo.
    Retorna um DataFrame ordenado do melhor (menor) para o pior.
    """
    df_fmt = df.copy()
    df_fmt = df_fmt.drop(columns="idade")  # remove coluna idade
    medias = df_fmt.mean().sort_values()
    return medias.reset_index().rename(columns={"index": "Modelo", 0: f"Média_{metrica}"})

# ============================
# Exemplos de uso
# ============================

# Médias para Feminino
medias_fem_rmse = medias_por_modelo(resultados[("Feminino", "RMSE")], "RMSE")
medias_fem_smpae = medias_por_modelo(resultados[("Feminino", "sMAPE")], "sMAPE")
medias_fem_mae = medias_por_modelo(resultados[("Feminino", "MAE")], "MAE")
print("Métricas globais Mulheres")
print(medias_fem_rmse)
print(medias_fem_smpae)
print(medias_fem_mae)

# Médias para Masculino
medias_mas_rmse = medias_por_modelo(resultados[("Masculino", "RMSE")], "RMSE")
medias_mas_smpae = medias_por_modelo(resultados[("Masculino", "sMAPE")], "sMAPE")
medias_mas_mae = medias_por_modelo(resultados[("Masculino", "MAE")], "MAE")
print("Métricas globais Homens")
print(medias_mas_rmse)
print(medias_mas_smpae)
print(medias_mas_mae)

# Médias para Ambos
medias_ambos_rmse = medias_por_modelo(resultados[("Ambos", "RMSE")], "RMSE")
medias_ambos_mae = medias_por_modelo(resultados[("Ambos", "MAE")], "MAE")
medias_ambos_smpae = medias_por_modelo(resultados[("Ambos", "sMAPE")], "sMAPE")
print("Métricas globais Total")
print(medias_ambos_rmse)
print(medias_ambos_smpae)
print(medias_ambos_mae)